In [22]:
import pandas as pd
import random   

url = "https://docs.google.com/spreadsheets/d/17yC9PcTkC3gY5orbQ5VQXeBDOStMx2Dfpo53cMZTn5k/export?format=xlsx"

df_tipos_de_vigas = pd.read_excel(url, sheet_name="Tipos de vigas")
df_capacidade_das_formas = pd.read_excel(url, sheet_name="Capacidade das fôrmas")

print(df_tipos_de_vigas)
print(df_capacidade_das_formas)

   Tipo  Tamanho (m)  Demanda
0     1         1.22       24
1     2         1.45       60
2     3         2.35       56
3     4         2.50       72
4     5         2.65       16
5     6         2.95       17
6     7         3.30       12
   Tipo de fôrma  Capacidade (m)
0              1           11.95
1              2           11.95
2              3           11.95
3              4           11.95
4              5           11.95
5              6           11.95
6              7            5.95


In [23]:
# 1. Vamos criar o nosso "chão" vazio
lista_oficial_de_pecas = []

# 2. Vamos ler a tabela de vigas, linha por linha
for indice, linha in df_tipos_de_vigas.iterrows():
    tamanho = linha['Tamanho (m)']
    demanda = int(linha['Demanda'])
    
    # 3. Vamos repetir o tamanho o número de vezes da demanda
    for vez in range(demanda):
        lista_oficial_de_pecas.append(tamanho)

# Só para a gente conferir se deu certo:
print(f"Total de peças no chão para cortar: {len(lista_oficial_de_pecas)} vigas.")

Total de peças no chão para cortar: 257 vigas.


In [24]:
# 4. Pegando o tamanho padrão da nossa fôrma (vamos pegar a maior da sua tabela)
tamanho_forma = df_capacidade_das_formas['Capacidade (m)'].max()

# 5. Configurando as regras do nosso jogo (GRASP)
alfa = 0.5            # Nosso nível de Área VIP (mistura os maiores com sorteio)
tentativas = 10       # Vamos começar com 10 vezes só para o código rodar rapidinho

# 6. O nosso "Caderno de Anotações" para guardar o recorde
melhor_solucao = []
menor_numero_de_formas = 999999 # Começa gigante para ser fácil de bater

print(f"O tamanho da nossa fôrma base será: {tamanho_forma}m")

O tamanho da nossa fôrma base será: 11.95m


In [25]:
# 7. Ligando o motor! Vamos fazer as 10 tentativas.
for tentativa in range(tentativas):
    print(f"\n--- Tentativa {tentativa + 1} de {tentativas} ---")
    
    # Derramamos as 257 peças no chão de novo
    pecas_no_chao = lista_oficial_de_pecas.copy()
    formas_desta_tentativa = [] # Para guardar as fôrmas desta rodada
    
    # 8. Enquanto ainda tiver peça solta no chão...
    while len(pecas_no_chao) > 0:
        
        forma_atual = [] # Pega uma fôrma nova, vazia
        espaco_sobrando = tamanho_forma 
        numero_forma = len(formas_desta_tentativa) + 1
        print(f"\nNova forma {numero_forma}: espaço inicial {espaco_sobrando}m")
        
        # 9. Vamos tentar colocar peças dentro dessa fôrma até não caber mais
        while True:
            
            # A) Quem cabe no espaço que sobrou?
            cabem = []
            for peca in pecas_no_chao:
                if peca <= espaco_sobrando:
                    cabem.append(peca)
                    
            print(f"Peças que cabem no espaço restante ({espaco_sobrando}m): {len(cabem)}")

            # Se a lista de quem cabe estiver vazia, essa fôrma lotou!
            if len(cabem) == 0:
                print(f"Forma {numero_forma} lotou. Conteúdo: {forma_atual} | Espaço sobrando: {espaco_sobrando}m")
                break # Para de tentar encher e sai deste bloquinho
                
            # B) Criando a Área VIP (RCL)
            maior_peca = max(cabem)
            menor_peca = min(cabem)
            nota_de_corte = maior_peca - alfa * (maior_peca - menor_peca)
            
            area_vip = []
            for peca in cabem:
                if peca >= nota_de_corte:
                    area_vip.append(peca)
                    
            print(f"Maior peça: {maior_peca}, menor peça: {menor_peca}, nota de corte: {nota_de_corte:.2f}")
            print(f"Área VIP tem {len(area_vip)} peças: {area_vip}")
                    
            # C) O Sorteio
            peca_sorteada = random.choice(area_vip)
            print(f"Peça sorteada para a forma {numero_forma}: {peca_sorteada}m")
            
            # D) Guardando a peça
            forma_atual.append(peca_sorteada) # Coloca na fôrma
            pecas_no_chao.remove(peca_sorteada) # Tira do chão
            espaco_sobrando = espaco_sobrando - peca_sorteada # Atualiza o espaço
            print(f"Forma {numero_forma} agora tem {len(forma_atual)} peças, espaço sobrando {espaco_sobrando}m, peças restantes no chão: {len(pecas_no_chao)}")
            
        # Quando a fôrma lota (pelo 'break' ali em cima), guardamos ela na lista
        formas_desta_tentativa.append(forma_atual)
        
    # 10. A tentativa acabou (não tem mais peça no chão). Fomos bem?
    quantidade_usada = len(formas_desta_tentativa)
    print(f"Tentativa {tentativa + 1}: usadas {quantidade_usada} fôrmas")
    
    # Se usamos menos fôrmas que o recorde anterior, atualizamos o placar!
    if quantidade_usada < menor_numero_de_formas:
        menor_numero_de_formas = quantidade_usada
        melhor_solucao = formas_desta_tentativa
        print(f"Novo recorde: {menor_numero_de_formas} fôrmas")

# 11. Depois que as 10 tentativas terminam, vamos ver o resultado
print(f"\nSucesso! O menor número de fôrmas necessárias foi: {menor_numero_de_formas}")
print(f"Melhor solução (primeiras 10 fôrmas): {melhor_solucao[:10]}")


--- Tentativa 1 de 10 ---

Nova forma 1: espaço inicial 11.95m
Peças que cabem no espaço restante (11.95m): 257
Maior peça: 3.3, menor peça: 1.22, nota de corte: 2.26
Área VIP tem 173 peças: [np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35), np.float64(2.35